IMPORTS AND GLOBAL CONFIGS

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone

LOADING THE DATASET

In [2]:
df = pd.read_csv("dataset/Dataset of Diabetes .csv")

print("Dataset Shape:", df.shape)

Dataset Shape: (1000, 14)


# DATA CLEANING AND STANDARDIZATION

SANITY FIX, EVEN THO I KNOW NO MISSING VALUES AND DUPES

In [3]:
missing_values = df.isnull().sum()

print(missing_values)
print("Total Missing Values:", missing_values.sum())

df = df.drop_duplicates()

ID           0
No_Pation    0
Gender       0
AGE          0
Urea         0
Cr           0
HbA1c        0
Chol         0
TG           0
HDL          0
LDL          0
VLDL         0
BMI          0
CLASS        0
dtype: int64
Total Missing Values: 0


STANDARDIZING COLUMN VALUES

In [4]:
df["CLASS"] = df["CLASS"].astype(str).str.strip().str.upper()

df["Gender"] = df["Gender"].astype(str).str.strip().str.upper()

print(df["CLASS"].value_counts())
print(df["Gender"].value_counts())

CLASS
Y    844
N    103
P     53
Name: count, dtype: int64
Gender
M    565
F    435
Name: count, dtype: int64


REMOVING IDENTIFIER COLUMNS, IRRELEVANT COLUMNS FOR THE MODELING

In [5]:
id_columns = ["ID", "No_Pation"]

df_model = df.drop(columns=id_columns)

#SEPARATING FEATURES AND TARGET

X = df_model.drop(columns=["CLASS"])
y = df_model["CLASS"]

# FEATURE ENGINEERING:

CATEGORIZING AND BINNING BMI AND AGE TO CAPTURE NON-LINEAR RELATIONSHIPS

In [6]:
#BMI BINNING

bmi_bins = [0, 18.5, 24.9, 29.9, np.inf]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese"]

X["BMI_Category"] = pd.cut(
    X["BMI"],
    bins=bmi_bins,
    labels=bmi_labels,
    include_lowest=True
)

#AGE BINNING

age_bins = [0, 30, 50, 70, np.inf]
age_labels = ["Young", "Middle-aged", "Senior", "Elderly"]

X["Age_Group"] = pd.cut(
    X["AGE"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

#CHECKING THE NEW BINNED FEATURES

print(X["BMI_Category"].value_counts())
print(X["Age_Group"].value_counts())

BMI_Category
Obese          526
Overweight     284
Normal         190
Underweight      0
Name: count, dtype: int64
Age_Group
Senior         742
Middle-aged    211
Young           27
Elderly         20
Name: count, dtype: int64


TARGET ENCODING

In [7]:
target_mapping = {
    "N": 0,
    "P": 1,
    "Y": 2
}

y_encoded = y.map(target_mapping)

# SPLITTING THE DATASET INTO TRAIN, VAL AND TEST SETS

In [8]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.125,
    random_state=42,
    stratify=y_temp
)

print("Training Shape:", X_train.shape)
print("Validation Shape:", X_val.shape)
print("Testing Shape:", X_test.shape)

print("Training Distribution:")
print(y_train.value_counts(normalize=True))

print("Validation Distribution:")
print(y_val.value_counts(normalize=True))

print("Testing Distribution:")
print(y_test.value_counts(normalize=True))

Training Shape: (700, 13)
Validation Shape: (100, 13)
Testing Shape: (200, 13)
Training Distribution:
CLASS
2    0.844286
0    0.102857
1    0.052857
Name: proportion, dtype: float64
Validation Distribution:
CLASS
2    0.84
0    0.10
1    0.06
Name: proportion, dtype: float64
Testing Distribution:
CLASS
2    0.845
0    0.105
1    0.050
Name: proportion, dtype: float64


In [9]:
numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['AGE', 'Urea', 'Cr', 'HbA1c', 'Chol', 'TG', 'HDL', 'LDL', 'VLDL', 'BMI']
Categorical features: ['Gender', 'BMI_Category', 'Age_Group']


# DATA PREPPING

IMPUTING AND NORMALIZING NUMERICAL (CONTINUOS) FEATURES

In [10]:
numeric_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

ENCODING CATEGORICAL FEATURES

In [11]:
categorical_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

COLUMN TRANSFORMER

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transform,
            numeric_features
        ),
        (
            "cat",
            categorical_transform,
            categorical_features
        )
    ]
)

TRANSFORMING THE SPLITS

In [13]:
def as_dense_array(data):
    """Return a dense NumPy array whether sklearn emits dense or sparse output."""
    return data.toarray() if hasattr(data, "toarray") else np.asarray(data)


X_train_prep = as_dense_array(preprocessor.fit_transform(X_train))
X_val_prep = as_dense_array(preprocessor.transform(X_val))
X_test_prep = as_dense_array(preprocessor.transform(X_test))

print("X_train_processed shape:", X_train_prep.shape)
print("X_val_processed shape:", X_val_prep.shape)
print("X_test_processed shape:", X_test_prep.shape)

#FULL DATASET PREP FOR CLUSTERING

full_preprocessor = clone(preprocessor)
X_full_prep = as_dense_array(full_preprocessor.fit_transform(X))

print("X_full_processed shape:", X_full_prep.shape)

X_train_processed shape: (700, 19)
X_val_processed shape: (100, 19)
X_test_processed shape: (200, 19)
X_full_processed shape: (1000, 19)


In [14]:
feature_names = preprocessor.get_feature_names_out()

X_train_transformed = pd.DataFrame(
    X_train_prep,
    columns=feature_names
)

print(X_train_transformed.head())

   num__AGE  num__Urea   num__Cr  num__HbA1c  num__Chol   num__TG  num__HDL  \
0  0.173265  -0.615806 -0.205505   -0.107281  -0.638297  0.197476  0.004366   
1 -0.181026   0.386181  0.310099    0.089968   2.815097 -1.093889  7.938003   
2  0.173265  -0.146125 -0.559983    0.642263   0.205866  1.560585 -0.877150   
3  0.173265  -0.333997 -0.559983    2.220250   0.436092 -0.519949  0.592042   
4 -0.299123   0.041748 -0.318293    0.878961   0.589576  2.278010 -0.730230   

   num__LDL  num__VLDL  num__BMI  cat__Gender_F  cat__Gender_M  \
0 -0.785889  -0.024392  0.065212            0.0            1.0   
1 -1.143925  -0.139180  0.675378            1.0            0.0   
2  0.109201   0.061698  0.878767            1.0            0.0   
3  0.467237  -0.311361  0.675378            1.0            0.0   
4  0.019692   0.061698  0.268601            0.0            1.0   

   cat__BMI_Category_Normal  cat__BMI_Category_Obese  \
0                       0.0                      1.0   
1               

SAVING PREPARED ARRAYS

In [15]:
processed_dir = Path("processed_arrays")
processed_dir.mkdir(exist_ok=True)

arrays_to_save = {
    "X_train.npy": X_train_prep,
    "X_val.npy": X_val_prep,
    "X_test.npy": X_test_prep,
    "y_train.npy": y_train.to_numpy(dtype=np.int64),
    "y_val.npy": y_val.to_numpy(dtype=np.int64),
    "y_test.npy": y_test.to_numpy(dtype=np.int64),
    "X_full.npy": X_full_prep,
    "y_full.npy": y_encoded.to_numpy(dtype=np.int64),
    "feature_names.npy": np.asarray(feature_names, dtype=str),
    "target_classes.npy": np.array(["N", "P", "Y"], dtype=str),
}

for filename, array in arrays_to_save.items():
    np.save(processed_dir / filename, array)

print(f"Saved {len(arrays_to_save)} NumPy arrays to {processed_dir.resolve()}")

Saved 10 NumPy arrays to C:\Users\mahar\PycharmProjects\PythonProject2\processed_arrays
